# Notebook 1 — Data Ingestion & Preparation

**Module:** 7006SCN Machine Learning and Big Data — Coventry University  
**Project:** Scalable Fake News Detection Using Distributed ML on Common Crawl  

---

## Objectives
1. Generate a realistic fake news dataset (simulating Common Crawl WET→filter output).
2. Validate schema with `StructType`.
3. Handle missing values and corrupted records.
4. Write cleaned data to **Parquet** with partitioning strategy.
5. Demonstrate persist/unpersist and Spark UI monitoring.

> **Note:** This dataset simulates Common Crawl output. Same Spark code runs on `s3://commoncrawl/` WET files — only the source changes.

> **Data Engineering rubric (20 %):** Schema enforcement, fault tolerance, storage-format decisions, partitioning.

In [ ]:
# ── 1.1  Bootstrap Spark ────────────────────────────────────────────────
import os, sys

# Environment variables for Spark on Windows
os.environ["JAVA_HOME"]             = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.16.8-hotspot"
os.environ["HADOOP_HOME"]           = r"C:\hadoop"
os.environ["PATH"]                  = r"C:\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FakeNewsDetection")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print(f"Spark version : {spark.version}")
print(f"Spark UI      : {sc.uiWebUrl}")

KeyboardInterrupt: 

In [ ]:
# ── 1.2  Generate Fake News Dataset ─────────────────────────────────────
# ~21K real + ~23K fake articles with realistic text patterns
import pandas as pd, numpy as np
from pathlib import Path

DATA_DIR = Path("../data/raw"); DATA_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(42)

reliable_sources = ["reuters.com","apnews.com","bbc.com","nytimes.com","washingtonpost.com","theguardian.com","npr.org"]
unreliable_sources = ["infowars.com","naturalnews.com","beforeitsnews.com","worldnewsdailyreport.com","newspunch.com"]
subjects_real = ["politicsNews","worldnews","business","science","technology"]
subjects_fake = ["News","politics","Government News","left-news","US_News"]

real_phrases = [
    "According to official reports released today, {topic} experts confirmed that recent developments in {area} have led to significant policy changes. The data suggests a consistent trend toward {direction}, with analysts noting historical patterns support this. Government officials stated new measures will address ongoing challenges in {sector}.",
    "A comprehensive study published in a peer-reviewed journal found that {topic} has shown measurable improvements over the past decade. Researchers analyzed data from multiple sources and concluded evidence-based approaches yield positive results in {area} and {sector}.",
    "Officials announced today that {topic} legislation has been passed following bipartisan negotiations. The bill addresses key concerns in {area} and {sector}, with provisions for increased funding. Economic analysts predict moderate positive impact on {direction}.",
    "International diplomats gathered to discuss {topic} challenges facing the global community. Representatives from over 40 countries produced a joint statement on {area} commitments with specific timelines for {sector} improvements.",
    "The quarterly economic report shows {topic} indicators trending {direction} for the third consecutive period. Labor market data reveals steady progress in {area}, with financial markets responding to {sector} developments.",
]
fake_phrases = [
    "BREAKING: Shocking revelations about {topic} that mainstream media REFUSES to report! A massive conspiracy involving {area} goes all the way to the top. Share before they DELETE it! Millions deceived about {sector}.",
    "You wont believe what they discovered about {topic}!!! Secret LEAKED documents show everything about {area} is a lie. The establishment hides the truth about {sector}. Wake up people!",
    "EXPOSED: Hidden truth about {topic} THEY dont want you to see. Anonymous insiders expose {area} cover-ups. This changes everything about {sector}. Evidence is UNDENIABLE.",
    "URGENT: New evidence proves {topic} scandal far WORSE than imagined. Deep state manipulating {area} for decades. Patriots must fight lies about {sector}. Forward to everyone!",
    "BOMBSHELL: What they HIDE about {topic} will SHOCK you. Whistleblowers expose {area} fraud at unprecedented scale. Cover-up reaches highest levels of {sector}. Media BLACKOUT proves complicity.",
]
topics = ["healthcare","economy","climate","education","immigration","technology","defense","trade","energy","infrastructure"]
areas = ["policy","reform","regulation","research","development","spending","cooperation","planning","assessment","governance"]
sectors = ["public health","financial markets","environmental protection","national security","international relations","digital privacy"]
directions = ["improvement","growth","stabilization","recovery","expansion"]
dates = pd.date_range("2017-01-01","2018-12-31",freq="D").strftime("%B %d, %Y").tolist()

def gen(templates, sources, subjects, n):
    rows = []
    for i in range(n):
        t1, t2 = np.random.choice(templates, 2, replace=True)
        kw = lambda: dict(topic=np.random.choice(topics), area=np.random.choice(areas),
                          sector=np.random.choice(sectors), direction=np.random.choice(directions))
        rows.append({"title": f"Article-{np.random.choice(topics)}-{i}", "text": t1.format(**kw())+" "+t2.format(**kw()),
                      "subject": np.random.choice(subjects), "date": np.random.choice(dates), "source": np.random.choice(sources)})
    return pd.DataFrame(rows)

true_pd = gen(real_phrases, reliable_sources, subjects_real, 21000)
fake_pd = gen(fake_phrases, unreliable_sources, subjects_fake, 23000)
true_pd.to_csv(DATA_DIR/"True.csv", index=False)
fake_pd.to_csv(DATA_DIR/"Fake.csv", index=False)
print(f"Real: {len(true_pd):,} | Fake: {len(fake_pd):,} | Total: {len(true_pd)+len(fake_pd):,}")

In [ ]:
# ── 1.3  Combine & label → Pandas ───────────────────────────────────────
true_pd["label"] = 0
fake_pd["label"] = 1
combined_pd = pd.concat([true_pd, fake_pd], ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f"Combined: {len(combined_pd):,} articles")
combined_pd.head(3)

In [ ]:
# ── 1.4  Schema enforcement (StructType) → Spark DataFrame ─────────────
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

ARTICLE_SCHEMA = StructType([
    StructField("title",   StringType(),  nullable=True),
    StructField("text",    StringType(),  nullable=False),
    StructField("subject", StringType(),  nullable=True),
    StructField("date",    StringType(),  nullable=True),
    StructField("source",  StringType(),  nullable=True),
    StructField("label",   IntegerType(), nullable=False),
])

raw_df = spark.createDataFrame(combined_pd, schema=ARTICLE_SCHEMA)
print(f"Spark DF: {raw_df.count():,} rows | Partitions: {raw_df.rdd.getNumPartitions()}")
raw_df.printSchema()
raw_df.show(3, truncate=80)

In [ ]:
# ── 1.5  Handle missing values & corrupted records ──────────────────────
from pyspark.sql import functions as F

print("=== Null / Empty audit ===")
for c in raw_df.columns:
    n = raw_df.filter(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == "")).count()
    print(f"  {c:15s} → {n:>6,} nulls/empty")

clean_df = (raw_df
    .dropna(subset=["text"])
    .filter(F.length("text") >= 100)
    .dropDuplicates(["text"])
    .fillna({"title":"Unknown","subject":"general","date":"unknown","source":"unknown"})
)
print(f"\nBefore: {raw_df.count():>10,}")
print(f"After:  {clean_df.count():>10,}")

In [ ]:
# ── 1.6  Persist + Write Parquet ────────────────────────────────────────
from pyspark import StorageLevel

clean_df.persist(StorageLevel.MEMORY_AND_DISK)
row_count = clean_df.count()
print(f"Cached {row_count:,} rows | Spark UI: {sc.uiWebUrl}")

OUTPUT_PATH = "../data/parquet/news_articles"
clean_df.repartition("subject").write.mode("overwrite").partitionBy("subject").parquet(OUTPUT_PATH)
print(f"✓ Parquet → {OUTPUT_PATH}")

verify_df = spark.read.parquet(OUTPUT_PATH)
print(f"  Rows: {verify_df.count():,} | Parts: {verify_df.rdd.getNumPartitions()} | Subjects: {verify_df.select('subject').distinct().count()}")

In [ ]:
# ── 1.7  Unpersist & Summary ───────────────────────────────────────────
clean_df.unpersist()
summary = verify_df.groupBy("label").count().toPandas()
summary["label_name"] = summary["label"].map({0: "Reliable", 1: "Fake"})
print(summary.to_string(index=False))
print(f"\n✓ Notebook 1 complete — {row_count:,} articles → Parquet.")